# CASE 1: Inference Reproducibility with Original Dataset #

Because the number of classification folds across the 100 runs is too large, we only provide the Jupyter notebook for one fold here, namely the first fold of the reshuffling process. This notebook reproduces the zero setting for the Meta-learner model. To reproduce the other results, simply follow the same workflow and continue running the remaining folds.

## classification

### PanPep

##### majority

In [ ]:
import sys

SCRIPT_DIR = "./"
TEST_DATA = "../../data/fig2/reshuffling_majority_0.csv"
NEGATIVE_DATA = "../../data/Control_dataset.txt"  #Blank file

!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --support_dir "../../data/majority_paper" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "{SCRIPT_DIR}/CASE1_majority"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: EAAGIGILTV on device: cuda:0
Skipping peptide EAAGIGILTV - result file already exists

Processing peptide: ELAGIGILTV on device: cuda:0
Skipping peptide ELAGIGILTV - result file already exists

Processing peptide: LPRRSGAAGA on device: cuda:0
Skipping peptide LPRRSGAAGA - result file already exists

Processing peptide: TPRVTGGGAM on device: cuda:0
Skipping peptide TPRVTGGGAM - result file already exists

Processing peptide: ATDALMTGY on device: cuda:0
Skipping peptide ATDALMTGY - result file already exists

Processing peptide: CINGVCWTV on device: cuda:0
Skipping peptide CINGVCWTV - result file already exists

Processing peptide: CRVLCCYVL on device: cuda:0
Skipping peptide CRVLCCYVL - result file already exists

Processing peptide: FPRPWLHGL on device: cuda:0
Skipping peptide FPRPWLHGL - result file already exists

Processing peptide: FRCPRRFCF on device: cuda:0
Skipping peptide FRCPRRFCF - result file already e

In [ ]:
import os
import pandas as pd

PARQUET_DIR = "./CASE1_majority"
CSV_PATH = "../../data/fig2/reshuffling_majority_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE1_majority_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] ATDALMTGY: 120 rows, 120 scored
[OK] CINGVCWTV: 488 rows, 488 scored
[OK] CRVLCCYVL: 158 rows, 158 scored
[OK] EAAGIGILTV: 78 rows, 78 scored
[OK] ELAGIGILTV: 2204 rows, 2204 scored
[OK] FPRPWLHGL: 80 rows, 80 scored
[OK] FRCPRRFCF: 96 rows, 96 scored
[OK] GILGFVFTL: 7192 rows, 7192 scored
[OK] GLCTLVAML: 934 rows, 934 scored
[OK] LLWNGPMAV: 90 rows, 90 scored
[OK] LPRRSGAAGA: 880 rows, 880 scored
[OK] NLVPMVATV: 7668 rows, 7668 scored
[OK] RAKFKQLL: 2524 rows, 2524 scored
[OK] RLRAEAQVK: 182 rows, 182 scored
[OK] SFHSLHLLF: 74 rows, 74 scored
[OK] TPQDLNTML: 82 rows, 82 scored
[OK] TPRVTGGGAM: 226 rows, 226 scored
[OK] VTEHDTLLY: 34 rows, 34 scored
[OK] YVLDHLIVV: 680 rows, 680 scored

Saved to ./result/CASE1_majority_scored.csv
Total rows: 23790, scored: 23790


In [40]:
!{sys.executable} ../../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE1_majority_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [41]:
import pandas as pd

df = pd.read_csv("./CASE1_majority_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE1_majority_scored.csv,0.54182,0.544373,success
1,./result,[FOLDER_SUMMARY],0.54182,0.544373,success (averaged over 1 files)


##### few-shot

In [ ]:
import sys

SCRIPT_DIR = "./"
TEST_DATA = "../../data/fig2/reshuffling_few_0.csv"
NEGATIVE_DATA = "../../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_few_shot.py \
    --mode single \
    --model default\
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --support_dir "../../data/few-paper" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "{SCRIPT_DIR}/CASE1_few"

Using 1 GPUs: [0]
Using model configuration: default
Inference mode: single
support_dir enabled: will load k-shot data from ../../data/few-paper

Processing peptide: RKTVRARSRTPSCRSRSHTPSRRRR on device: cuda:0
Positive TCRs: 4, Negative TCRs: 3386
Total batches: 1
Loading k-shot data from: ../../data/few-paper
Query data size: 3390

Processing batch 1/1 for peptide RKTVRARSRTPSCRSRSHTPSRRRR - 2026-03-16 01:48:50
Finetuning model...
batch processing time: 0.72s, Progress: 100.0%
Successfully wrote 3390 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/.//CASE1_few/RKTVRARSRTPSCRSRSHTPSRRRR.parquet

Total time for peptide RKTVRARSRTPSCRSRSHTPSRRRR: 0.75s

Processing peptide: APRGPHGGAASGL on device: cuda:0
Positive TCRs: 12, Negative TCRs: 3386
Total batches: 1
Loading k-shot data from: ../../data/few-paper
Query data size: 3398

Processing batch 1/1 for peptide APRGPHGGAASGL - 2026-03-16 01:48:50
Finetuning model...
batch processing time: 0.25s, Progress: 100.0%
Suc

In [ ]:
import os
import pandas as pd

PARQUET_DIR = "./CASE1_few"
CSV_PATH = "../../data/fig2/reshuffling_few_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE1_few_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] AAFKRSCLK: 6 rows, 6 scored
[OK] AEMEFCPCC: 28 rows, 28 scored
[OK] ALIHHNTHL: 4 rows, 4 scored
[OK] ALIHHNTYL: 4 rows, 4 scored
[OK] ALSYTPAEV: 4 rows, 4 scored
[OK] AMAGSLVFL: 2 rows, 2 scored
[OK] APARLERRHSA: 4 rows, 4 scored
[OK] APRGPHGGAASGL: 12 rows, 12 scored
[OK] AVGSHVYSV: 2 rows, 2 scored
[OK] AVYDGREHTV: 4 rows, 4 scored
[OK] CCIEVINNT: 12 rows, 12 scored
[OK] CFLLILPNC: 24 rows, 24 scored
[OK] CHIPVEPNT: 8 rows, 8 scored
[OK] CLGGLLTMV: 180 rows, 180 scored
[OK] CLLGTYTQDV: 4 rows, 4 scored
[OK] CLLWSFQTSA: 22 rows, 22 scored
[OK] EIQEFCPNT: 28 rows, 28 scored
[OK] ESEERPPTPY: 4 rows, 4 scored
[OK] FDVMVMPNL: 8 rows, 8 scored
[OK] FIDCYLLAI: 2 rows, 2 scored
[OK] FIDQYYSSI: 2 rows, 2 scored
[OK] FIFSYVVAV: 2 rows, 2 scored
[OK] FILDAVQRV: 2 rows, 2 scored
[OK] FIYQYYSSI: 4 rows, 4 scored
[OK] FLDEFMEAV: 2 rows, 2 scored
[OK] FLFMYLVTV: 4 rows, 4 scored
[OK] FLFTFFASI: 2 rows, 2 scored
[OK] FLKEMGGL: 4 rows, 4 scored
[OK] FLKEQGGL: 4 rows, 4 scored
[OK] FLKETGGL: 4 ro

In [43]:
!{sys.executable} ../../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE1_few_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [44]:
import pandas as pd

df = pd.read_csv("./CASE1_few_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE1_few_scored.csv,0.633369,0.654649,success
1,./result,[FOLDER_SUMMARY],0.633369,0.654649,success (averaged over 1 files)


#### zero-shot

In [ ]:
import sys

SCRIPT_DIR = "./"
TEST_DATA = "../../data/fig2/reshuffling_zero_0.csv"
NEGATIVE_DATA = "../../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_zero_shot.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --result_dir "{SCRIPT_DIR}/CASE1_zero"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --model default

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: SWISDIRAGTAPLCRNHIKSSCSLI on device: cuda:0

Processing batch 1/1 for peptide SWISDIRAGTAPLCRNHIKSSCSLI - 2026-03-16 02:17:07
batch processing time: 0.99s, Progress: 100.0%
Successfully wrote 9556 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/.//CASE1_zero/SWISDIRAGTAPLCRNHIKSSCSLI.parquet

Peptide SWISDIRAGTAPLCRNHIKSSCSLI processing time: 1.03s

Processing peptide: APAGPHGGAASGL on device: cuda:0

Processing batch 1/1 for peptide APAGPHGGAASGL - 2026-03-16 02:17:08
batch processing time: 0.65s, Progress: 100.0%
Successfully wrote 9556 records to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/.//CASE1_zero/APAGPHGGAASGL.parquet

Peptide APAGPHGGAASGL processing time: 0.68s

Processing peptide: APRGAHGGAASGL on device: cuda:0

Processing batch 1/1 for peptide APRGAHGGAASGL - 2026-03-16 02:17:09
batch processing time: 0.64s, Progress: 100.0%
Successfully wrote 9556 records to 

In [49]:
import os
import pandas as pd

PARQUET_DIR = "./CASE1_zero"
CSV_PATH = "../../data/fig2/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE1_zero_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] ACDPHSGHFV: 4 rows, 4 scored
[OK] AIMDKNIIL: 4 rows, 4 scored
[OK] ALGGLLTMV: 4 rows, 4 scored
[OK] ALGIGILTV: 98 rows, 98 scored
[OK] ALLETLSLLL: 4 rows, 4 scored
[OK] ALLQVTLLL: 4 rows, 4 scored
[OK] ALSYTPVEV: 4 rows, 4 scored
[OK] ALVGAIPSI: 4 rows, 4 scored
[OK] ALVPMVATV: 4 rows, 4 scored
[OK] ALWGFFPVL: 6 rows, 6 scored
[OK] ALWGPDPAA: 4 rows, 4 scored
[OK] AMAGSPVFL: 4 rows, 4 scored
[OK] AMAVLYLAL: 4 rows, 4 scored
[OK] APAGPHGGAASGL: 4 rows, 4 scored
[OK] APQPAPENAY: 4 rows, 4 scored
[OK] APRGAHGGAASGL: 4 rows, 4 scored
[OK] APRGPAGGAASGL: 4 rows, 4 scored
[OK] ASFRPELAEFW: 4 rows, 4 scored
[OK] ASFSPELRMAW: 4 rows, 4 scored
[OK] CISSCNPNL: 4 rows, 4 scored
[OK] CLAGLLTMV: 4 rows, 4 scored
[OK] CLGALLTMV: 4 rows, 4 scored
[OK] CLGGLATMV: 4 rows, 4 scored
[OK] CLGGLLAMV: 4 rows, 4 scored
[OK] DLAGIGILTV: 6 rows, 6 scored
[OK] EEAAGIGIL: 6 rows, 6 scored
[OK] EFFWDANDIY: 4 rows, 4 scored
[OK] EFTVSGNIL: 6 rows, 6 scored
[OK] EHEGSGPEL: 4 rows, 4 scored
[OK] EIAGIGILTV: 6 r

In [ ]:
!{sys.executable} ../../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE1_zero_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [ ]:
import pandas as pd

df = pd.read_csv("./CASE1_zero_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE1_zero_scored.csv,0.506968,0.531179,success
1,./result,[FOLDER_SUMMARY],0.506968,0.531179,success (averaged over 1 files)


#### Meta-learner

In [53]:
import sys

SCRIPT_DIR = "./"
TEST_DATA = "../../data/fig2/reshuffling_zero_0.csv"
NEGATIVE_DATA = "../../data/Control_dataset.txt"

!{sys.executable} {SCRIPT_DIR}/inference_meta_learner.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --result_dir "{SCRIPT_DIR}/CASE1_meta"\
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" 

Using 1 GPUs: [0]
GPU 0 will process 260 peptides with approximately 1370 TCRs

Processing peptide: ALGIGILTV on device: cuda:0
Positive TCRs: 98, Negative TCRs: 9553
Total batches: 1
Query data size: 9651

Processing batch 1/1 for peptide ALGIGILTV - 2026-03-16 02:25:37
batch processing time: 0.90s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/.//CASE1_meta/ALGIGILTV.parquet

Total time for peptide ALGIGILTV: 1.02s

Processing peptide: LLHGFSFYL on device: cuda:0
Positive TCRs: 72, Negative TCRs: 9553
Total batches: 1
Query data size: 9625

Processing batch 1/1 for peptide LLHGFSFYL - 2026-03-16 02:25:38
batch processing time: 0.70s, Progress: 100.0%
Successfully saved results to /mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/.//CASE1_meta/LLHGFSFYL.parquet

Total time for peptide LLHGFSFYL: 0.81s

Processing peptide: YLQQNWWTL on device: cuda:0
Positive TCRs: 36, Negative TCRs: 9553
Total batches: 1
Query data

In [58]:
import os
import pandas as pd

PARQUET_DIR = "./CASE1_meta"
CSV_PATH = "../../data/fig2/reshuffling_zero_0.csv"

OUTPUT_DIR = "./result"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "CASE1_meta_scored.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_df = pd.read_csv(CSV_PATH)

if "label" in csv_df.columns:
    csv_df = csv_df.rename(columns={"label": "Label"})

peptides = csv_df["peptide"].unique()

results = []
for pep in peptides:
    parquet_path = os.path.join(PARQUET_DIR, f"{pep}.parquet")
    if not os.path.exists(parquet_path):
        print(f"[SKIP] No parquet for {pep}")
        continue

    parquet_df = pd.read_parquet(parquet_path)[["CDR3", "Score"]].rename(
        columns={"CDR3": "binding_TCR"}
    )

    pep_rows = csv_df[csv_df["peptide"] == pep].copy()
    merged = pep_rows.merge(parquet_df, on="binding_TCR", how="left")
    results.append(merged)

    print(f"[OK] {pep}: {len(merged)} rows, {merged['Score'].notna().sum()} scored")

if results:
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(OUTPUT_PATH, index=False)

    print(f"\nSaved to {OUTPUT_PATH}")
    print(f"Total rows: {len(final_df)}, scored: {final_df['Score'].notna().sum()}")
else:
    print("No results to save.")

[OK] ACDPHSGHFV: 4 rows, 4 scored
[OK] AIMDKNIIL: 4 rows, 4 scored
[OK] ALGGLLTMV: 4 rows, 4 scored
[OK] ALGIGILTV: 98 rows, 98 scored
[OK] ALLETLSLLL: 4 rows, 4 scored
[OK] ALLQVTLLL: 4 rows, 4 scored
[OK] ALSYTPVEV: 4 rows, 4 scored
[OK] ALVGAIPSI: 4 rows, 4 scored
[OK] ALVPMVATV: 4 rows, 4 scored
[OK] ALWGFFPVL: 6 rows, 6 scored
[OK] ALWGPDPAA: 4 rows, 4 scored
[OK] AMAGSPVFL: 4 rows, 4 scored
[OK] AMAVLYLAL: 4 rows, 4 scored
[OK] APAGPHGGAASGL: 4 rows, 4 scored
[OK] APQPAPENAY: 4 rows, 4 scored
[OK] APRGAHGGAASGL: 4 rows, 4 scored
[OK] APRGPAGGAASGL: 4 rows, 4 scored
[OK] ASFRPELAEFW: 4 rows, 4 scored
[OK] ASFSPELRMAW: 4 rows, 4 scored
[OK] CISSCNPNL: 4 rows, 4 scored
[OK] CLAGLLTMV: 4 rows, 4 scored
[OK] CLGALLTMV: 4 rows, 4 scored
[OK] CLGGLATMV: 4 rows, 4 scored
[OK] CLGGLLAMV: 4 rows, 4 scored
[OK] DLAGIGILTV: 6 rows, 6 scored
[OK] EEAAGIGIL: 6 rows, 6 scored
[OK] EFFWDANDIY: 4 rows, 4 scored
[OK] EFTVSGNIL: 6 rows, 6 scored
[OK] EHEGSGPEL: 4 rows, 4 scored
[OK] EIAGIGILTV: 6 r

In [59]:
!{sys.executable} ../../metric_calculation/AUC.py \
                    --root_dir ./result \
                    --output_file ./CASE1_meta_auc.csv

Finding CSV files...
Found 1 CSV files
Processed 1/1 files (100.0%)
Failed: 1


In [60]:
import pandas as pd

df = pd.read_csv("./CASE1_meta_auc.csv")
df

,folder_path,file_name,roc_auc,pr_auc,status
0,./result,CASE1_meta_scored.csv,0.479802,0.524268,success
1,./result,[FOLDER_SUMMARY],0.479802,0.524268,success (averaged over 1 files)


### Other method

If you would like to reproduce other methods, you can use their web servers. For DLpTCR, you can use http://jianglab.org.cn/DLpTCR/ and select pTCRβ. For ERGO-II, you can use http://tcr2.cs.biu.ac.il/home and choose VDJdb mode.

## rank

For demonstration purposes, since running on the full background pool would be too time-consuming, we constructed a subset pool by randomly sampling 0.1% of the background library and used it for reproduction and result presentation. If you would like to reproduce the full original results, you can simply replace the subset background pool with the complete background library and rerun the same pipeline. In other words, the current setup is only for demonstration efficiency, while the full results can be recovered by using the original background pool.

In [ ]:
import sys

SCRIPT_DIR = "./"
TEST_DATA = "../../data/fig2/reshuffling_majority_0.csv"
NEGATIVE_DATA = "../../data/Combined_library_sample_0.1pct.txt"

!{sys.executable} {SCRIPT_DIR}/inference_majority.py \
    --gpu 0 \
    --distillation 3 \
    --batch_size 10000 \
    --test_data "{TEST_DATA}" \
    --negative_data "{NEGATIVE_DATA}" \
    --model_path "{SCRIPT_DIR}/Requirements" \
    --support_dir "../../data/majority_paper" \
    --peptide_encoding "{SCRIPT_DIR}/peptide_b.npz" \
    --tcr_encoding "{SCRIPT_DIR}/tcr_b.npz" \
    --result_dir "{SCRIPT_DIR}/CASE1_majority_rank"

Using 1 GPUs: [0]
Using model configuration: default

Processing peptide: EAAGIGILTV on device: cuda:0
Positive TCRs: 78, Negative TCRs: 57102
Total batches: 6
Loading support data from: ../../data/majority_paper
Loading support data for peptide: EAAGIGILTV
Query data size: 57179

Processing batch 1/6 for peptide EAAGIGILTV - 2026-03-16 03:01:19
Finetuning model...
batch processing time: 13.01s, Progress: 16.7%

Processing batch 2/6 for peptide EAAGIGILTV - 2026-03-16 03:01:32
Using finetuned model for inference...
batch processing time: 0.83s, Progress: 33.3%

Processing batch 3/6 for peptide EAAGIGILTV - 2026-03-16 03:01:33
Using finetuned model for inference...
batch processing time: 0.84s, Progress: 50.0%

Processing batch 4/6 for peptide EAAGIGILTV - 2026-03-16 03:01:34
Using finetuned model for inference...
batch processing time: 0.80s, Progress: 66.7%

Processing batch 5/6 for peptide EAAGIGILTV - 2026-03-16 03:01:34
Using finetuned model for inference...
batch processing time: 

In [64]:
!{sys.executable} ../../metric_calculation/sort.py \
                    --input_dir ./CASE1_majority_rank \
                    --output_dir ./CASE1_majority_sort

Found 19 files to process:
  CSV files: 0
  Parquet files: 19

Completed processing 19 files


In [77]:
!{sys.executable} ../../metric_calculation/bedroc.py \
                    -i ./CASE1_majority_sort/CASE1_majority_rank \
                    -o ./CASE1_majority_bedroc

2026-03-16 03:18:22,221 - INFO - Input directory: ./CASE1_majority_sort/CASE1_majority_rank
2026-03-16 03:18:22,222 - INFO - Output directory: ./CASE1_majority_bedroc
2026-03-16 03:18:22,222 - INFO - Number of processes: 8
Total: 19 files, To process: 19 files
Processing files:   0%|                                  | 0/19 [00:00<?, ?it/s]
File: EAAGIGILTV_sorted.parquet

File: FPRPWLHGL_sorted.parquet

File: ELAGIGILTV_sorted.parquet

File: CRVLCCYVL_sorted.parquet

File: CINGVCWTV_sorted.parquet

File: ATDALMTGY_sorted.parquet

File: GILGFVFTL_sorted.parquet

File: FRCPRRFCF_sorted.parquet
Processing files:   5%|█▎                        | 1/19 [00:00<00:03,  5.57it/s]
File: GLCTLVAML_sorted.parquet

File: LLWNGPMAV_sorted.parquet

File: LPRRSGAAGA_sorted.parquet

File: RAKFKQLL_sorted.parquet

File: RLRAEAQVK_sorted.parquet

File: NLVPMVATV_sorted.parquet

File: SFHSLHLLF_sorted.parquet

File: TPQDLNTML_sorted.parquet
Processing files:  63%|███████████████▊         | 12/19 [00:00<00

In [ ]:
import os
import glob
import pandas as pd

detailed_dir = "./CASE1_majority_bedroc"

# Print all CSV files in the detailed results directory
csv_files = sorted(glob.glob(os.path.join(detailed_dir, "*.csv")))

print(f"Found {len(csv_files)} CSV file(s) in: {detailed_dir}\n")

for file_path in csv_files:
    print("=" * 100)
    print(f"File: {file_path}")
    print("=" * 100)
    df = pd.read_csv(file_path)
    print(df)
    print("\n")

Found 20 CSV file(s) in: ./CASE1_majority_bedroc

File: ./CASE1_majority_bedroc/summary_by_directory.csv
                                   Directory  BEDROC_Mean  BEDROC_Std  \
0  ./CASE1_majority_sort/CASE1_majority_rank     0.200508    0.077391   

   File_Count  
0          19  


File: ./CASE1_majority_bedroc/temp_ATDALMTGY_sorted.csv
                                   Directory                  Filename  \
0  ./CASE1_majority_sort/CASE1_majority_rank  ATDALMTGY_sorted.parquet   

   BEDROC_Score  Alpha  
0      0.151686     20  


File: ./CASE1_majority_bedroc/temp_CINGVCWTV_sorted.csv
                                   Directory                  Filename  \
0  ./CASE1_majority_sort/CASE1_majority_rank  CINGVCWTV_sorted.parquet   

   BEDROC_Score  Alpha  
0      0.095342     20  


File: ./CASE1_majority_bedroc/temp_CRVLCCYVL_sorted.csv
                                   Directory                  Filename  \
0  ./CASE1_majority_sort/CASE1_majority_rank  CRVLCCYVL_sorted.parquet

In [84]:
!{sys.executable} "../../metric_calculation/success_rate&hit_rate.py" \
    --root_dir "./CASE1_majority_sort/CASE1_majority_rank" \
    --output "CASE1_majority_sort" \
    --output_dir "./CASE1_majority_success"

2026-03-16 03:33:16,918 - INFO - Multiprocessing start method set to 'spawn'.
2026-03-16 03:33:17,022 - INFO - Configuration: ProcessingConfig(root_dir='./CASE1_majority_sort/CASE1_majority_rank', top_k_ratio=1, batch_size=150, num_gpus=1, output_file='CASE1_majority_sort', output_dir='./CASE1_majority_success', gpu_id=None)
2026-03-16 03:33:17,022 - INFO - Detected 1 GPU(s)
Processing directories: 100%|█████████████████████| 1/1 [00:02<00:00,  2.31s/it]
2026-03-16 03:33:19,357 - INFO - Combining 1 DataFrames...
2026-03-16 03:33:19,600 - INFO - Final results saved to CASE1_majority_success/CASE1_majority_sort.csv and CASE1_majority_success/CASE1_majority_sort.parquet


In [85]:
!{sys.executable} "../../metric_calculation/get_success_AUC.py" \
    --input "./CASE1_majority_success/CASE1_majority_sort.parquet" \
    --output "CASE1_majority_success_AUC" 

Found 1 directories.
1. 'CASE1_majority_sort/CASE1_majority_rank'
['CASE1_majority_sort/CASE1_majority_rank'] n=64762  Top_K range=(1, 64762)  y range=(0.000232883, 1)
/mnt/d/PanPep_Reusability/inference/PanPep_Weight_Inference/../../metric_calculation/get_success_AUC.py:190: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area = float(np.trapz(ys, xs_norm))
  AUC@0.0100%: area=9.30964e-08
  AUC@0.1000%: area=6.1079e-06
  AUC@1.0000%: area=0.000316417
  AUC@5.0000%: area=0.00575419
  AUC@10.0000%: area=0.0189795
  AUC@20.0000%: area=0.0590135
  AUC@100.0000%: area=0.711741

Saved results to: CASE1_majority_success_AUC
